# Throttle validation on Kaggle dual NVIDIA Tesla T4 — documented execution notebook

This copy adds **Markdown descriptions only** around the original executed Kaggle cells.

**Preservation rule:** every original code cell, execution count, code source, cell metadata, and captured output is copied byte-for-byte at the JSON object level from `p8-kushagra-throttle.ipynb`. No original code or output was edited, re-executed, removed, or regenerated.

The added Markdown explains the purpose of each original cell and summarizes observed results only where those results are present in the original outputs.


### Cell 1 — Kaggle Python environment preamble
This is Kaggle's default starter cell. It documents that the notebook is running in Kaggle's managed Python container and points to the standard input/output filesystem conventions. It was not used to change the environment and produced no output.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### Cell 2 — Create the validation workspace
Creates `/kaggle/working/throttle-t4-validation` with separate `logs` and `results` directories, then records the UTC start time. This gives the experiment one dedicated evidence directory from the beginning.

In [1]:
%%bash
set -euxo pipefail

mkdir -p /kaggle/working/throttle-t4-validation
mkdir -p /kaggle/working/throttle-t4-validation/logs
mkdir -p /kaggle/working/throttle-t4-validation/results

date -u | tee /kaggle/working/throttle-t4-validation/logs/start-time.txt

Thu Aug 27 04:40:48 AM UTC 2026


+ mkdir -p /kaggle/working/throttle-t4-validation
+ mkdir -p /kaggle/working/throttle-t4-validation/logs
+ mkdir -p /kaggle/working/throttle-t4-validation/results
+ date -u
+ tee /kaggle/working/throttle-t4-validation/logs/start-time.txt


### Cell 3 — Record the untouched Kaggle hardware/software environment
Captures the OS, Python version, NVIDIA driver/GPU details, PyTorch/CUDA state, and CUDA compiler information into `logs/environment.txt`. The output establishes the dual-T4 hardware/software reproducibility baseline.

In [2]:
%%bash

OUT=/kaggle/working/throttle-t4-validation/logs/environment.txt

{
    echo "===== DATE ====="
    date -u

    echo
    echo "===== OS ====="
    uname -a
    cat /etc/os-release || true

    echo
    echo "===== PYTHON ====="
    which python
    python --version
    python -VV

    echo
    echo "===== NVIDIA-SMI ====="
    nvidia-smi

    echo
    echo "===== NVIDIA-SMI QUERY ====="
    nvidia-smi \
      --query-gpu=index,name,memory.total,compute_cap,driver_version \
      --format=csv

    echo
    echo "===== TORCH ====="
    python - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(
        i,
        p.name,
        "VRAM_GiB=", round(p.total_memory / 1024**3, 2),
        "CC=", f"{p.major}.{p.minor}",
    )
PY

    echo
    echo "===== NVCC ====="
    nvcc --version || true

} 2>&1 | tee "$OUT"

===== DATE =====
Thu Aug 27 04:40:58 AM UTC 2026

===== OS =====
Linux 6b87dff7c6cd 6.12.90+ #1 SMP Sat May 30 15:40:53 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy

===== PYTHON =====
/usr/local/bin/python
Python 3.12.13
Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

===== NVIDIA-SMI =====
Thu Aug 27 04:40:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  

### Cell 4 — Clone upstream Throttle
Clones `KushagraKanaujia/throttle` into `/kaggle/working/throttle`, then records the exact Git commit and repository status. This pins the external validation to a specific upstream source revision.

In [3]:
%%bash
set -euxo pipefail

cd /kaggle/working

rm -rf throttle

git clone https://github.com/KushagraKanaujia/throttle.git

cd throttle

git rev-parse HEAD
git status --short

7c470078de78f4771b190db366be391e9f8b6e44


+ cd /kaggle/working
+ rm -rf throttle
+ git clone https://github.com/KushagraKanaujia/throttle.git
Cloning into 'throttle'...
+ cd throttle
+ git rev-parse HEAD
+ git status --short


### Cell 5 — Save Throttle Git provenance
Writes the tested commit, branch, and latest commit summary to `logs/throttle-git.txt`. The observed commit is `7c470078de78f4771b190db366be391e9f8b6e44`.

In [4]:
%%bash

cd /kaggle/working/throttle

{
    echo "Throttle commit:"
    git rev-parse HEAD

    echo
    echo "Throttle branch:"
    git branch --show-current

    echo
    echo "Latest commit:"
    git log -1 --oneline

} | tee /kaggle/working/throttle-t4-validation/logs/throttle-git.txt

Throttle commit:
7c470078de78f4771b190db366be391e9f8b6e44

Throttle branch:
main

Latest commit:
7c47007 Add RunPod deployment checklist and throttle report command


### Cell 6 — Editable Throttle installation attempt
Installs the cloned Throttle source with `pip install -e .` in the Kaggle base Python environment. No virtual environment is activated; the purpose is to make the local checkout directly executable inside Kaggle.

In [13]:
%%bash
set -o pipefail

cd /kaggle/working/throttle
pip install -e .

Obtaining file:///kaggle/working/throttle
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for throttle-pro (pyproject.toml): started
  Building editable for throttle-pro (pyproject.toml): finished with status 'done'
  Created wheel for throttle-pro: filename=throttle_pro-0.3.0-0.editable-py3-none-any.whl size=16699 sha256=6e0d52560c523305edc3927b643a5db9d7aa571cf0a5357443efc6fa67c1469f
  Stored in directory: /tmp/pip-ephem-wheel-cache-pbas7bee/wheels/23/0d/34/e09155650ca477ff9ae51de1d93fec1764c0aed5d77537ba4f
Successfully buil

### Cell 7 — Install Throttle from the local repository
Runs `python -m pip install .`, saves the complete installation log, and checks `throttle --version`. This confirms an ordinary local package installation of Throttle 0.3.0.

In [14]:
%%bash
set -o pipefail

cd /kaggle/working/throttle


python -m pip install . 2>&1 | tee \
  /kaggle/working/throttle-t4-validation/logs/readme-02-install.log

throttle --version 2>&1 | tee \
  /kaggle/working/throttle-t4-validation/logs/readme-03-version.log

Processing /kaggle/working/throttle
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for throttle-pro: filename=throttle_pro-0.3.0-py3-none-any.whl size=188937 sha256=465dd64163d7ab3765cac21ff2fcb7b1f08ef461508e0cdf210b5d9bdbabade0
  Stored in directory: /tmp/pip-ephem-wheel-cache-2gfed75b/wheels/23/0d/34/e09155650ca477ff9ae51de1d93fec1764c0aed5d77537ba4f
Successfully built throttle-pro
  Attempting uninstall: throttle-pro
    Found existing installation: throttle-pro 0.3.0
    Uninstalling throttle-pro-0.3.0:
      Successfully uninstalled throttle-pro-0.3.0
throttle 0.3.0


### Cell 8 — Capture the Throttle CLI surface
Records the version, top-level help, `validate-sim --help`, and `experimental-tuning --help`. This preserves the exact CLI contract of the tested Throttle 0.3.0 revision.

In [15]:
%%bash

cd /kaggle/working/throttle


{
    throttle --version

    echo
    echo "===== THROTTLE HELP ====="
    throttle --help

    echo
    echo "===== VALIDATE-SIM HELP ====="
    throttle validate-sim --help

    echo
    echo "===== EXPERIMENTAL-TUNING HELP ====="
    throttle experimental-tuning --help

} 2>&1 | tee \
  /kaggle/working/throttle-t4-validation/logs/throttle-cli-help.txt

throttle 0.3.0

===== THROTTLE HELP =====
usage: throttle [-h] [--version]
                {plan,smoke,benchmark,diagnose,experimental-tuning,golden,compare,report,demo,cost,validate-sim,measure,proxy}
                ...

Safety-first measurements for an existing OpenAI-compatible endpoint.

positional arguments:
  {plan,smoke,benchmark,diagnose,experimental-tuning,golden,compare,report,demo,cost,validate-sim,measure,proxy}
    plan                show traffic, cost, destination, and privacy without
                        sending traffic
    smoke               run a short, explicitly non-decision-grade check
    benchmark           run sustained exploratory evidence blocks (sweeps are
                        not counterbalanced)
    diagnose            pre-flight bottleneck regime classification (not
                        decision-grade)
    experimental-tuning
                        run an opt-in, suggestion-only server-metrics analysis
    golden              orchestrate the si

### Cell 9 — Install kaggle-vllm 0.1.2
Installs `kaggle-vllm[hub]==0.1.2`, which provides the Kaggle-specific runtime bootstrap/staging used for the dual-T4 vLLM backend.

In [16]:
%pip install -q "kaggle-vllm[hub]==0.1.2" 

Note: you may need to restart the kernel to use updated packages.


### Cell 10 — Verify both CLIs after installation
Checks that Throttle remains available and displays the kaggle-vllm command surface, guarding against package-install side effects before native runtime setup.

In [17]:
%%bash

echo "===== THROTTLE ====="
throttle --version

echo
echo "===== KAGGLE-VLLM ====="
kaggle-vllm --help | head -40

===== THROTTLE =====
throttle 0.3.0

===== KAGGLE-VLLM =====
usage: kaggle-vllm [-h]
                   {doctor,fingerprint,build-env,bootstrap,env,verify-gpus,inspect-shards,verify-wheel,stage-wheel,serve}
                   ...

positional arguments:
  {doctor,fingerprint,build-env,bootstrap,env,verify-gpus,inspect-shards,verify-wheel,stage-wheel,serve}
    doctor              compare runtime with the validated profile
    fingerprint         print a JSON runtime fingerprint
    build-env           print validated source-build settings
    bootstrap           explicitly download and stage the validated Kaggle
                        native runtime
    env                 print activation exports from a completed bootstrap
                        manifest
    verify-gpus         verify T4/SM75 and TP size
    inspect-shards      inspect a persistent sharded_state directory
    verify-wheel        calculate/verify SHA256
    stage-wheel         stage wheel with pip --no-deps
    serve 

### Cell 11 — Dry-run the kaggle-vllm bootstrap
Runs `kaggle-vllm bootstrap --strict --dry-run` and records the compatibility/artifact plan without performing the full runtime staging operation.

In [18]:
%%bash
set -o pipefail

kaggle-vllm bootstrap --strict --dry-run 2>&1 | tee \
  /kaggle/working/throttle-t4-validation/logs/kaggle-vllm-bootstrap-dry-run.txt

profile: kaggle-t4x2-cu128
compatible: True
strict: True
HF repository: waqasm86/kaggle-vllm-binaries
immutable revision: f6b4f10de54924ed6fe9e28cceab84eca7276ab6
wheel: vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
expected SHA256: 5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c
cache: /kaggle/working/kaggle-vllm-cache
staged: /kaggle/working/vllm-staged
overlay: /kaggle/working/vllm-runtime-overlay
manifest: /kaggle/working/kaggle-vllm-runtime.json
PASS: Python implementation: CPython
PASS: Python ABI: cp312
PASS: operating system: Linux
PASS: machine: x86_64
PASS: Kaggle runtime: True
PASS: PyTorch: 2.10.0+cu128
PASS: PyTorch CUDA: 12.8
PASS: visible GPU count: 2
PASS: GPU model: Tesla T4, Tesla T4
PASS: GPU compute capability: SM75, SM75
PASS: NCCL: 2.27.5
dry-run: no download or filesystem changes performed
would run: /usr/bin/python3 -m pip install --target /kaggle/working/vllm-staged --no-deps --no-cache-dir /kaggle/working/kaggle-vllm

### Cell 12 — Bootstrap the native kaggle-vllm runtime
Performs the real strict bootstrap and saves its full log. This stages the validated native vLLM wheel and runtime overlay for the Kaggle T4 profile.

In [19]:
%%bash
set -o pipefail

kaggle-vllm bootstrap --strict 2>&1 | tee \
  /kaggle/working/throttle-t4-validation/logs/kaggle-vllm-bootstrap.txt

Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 51.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 146.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 210.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 327.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 316.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 314.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 305.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 344.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 336.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 355.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 344.1 MB/s eta 0:00:00
   ━━━━━━━━━━━

### Cell 13 — Run `kaggle-vllm doctor`
Activates the staged runtime in this shell and checks the Kaggle profile. The captured output reports a PASS with Python 3.12.13, PyTorch 2.10.0+cu128, CUDA usable, NCCL 2.27.5, and two Tesla T4 GPUs at compute capability 7.5.

In [21]:
%%bash
set -o pipefail

eval "$(kaggle-vllm env)"

kaggle-vllm doctor 2>&1 | tee \
  /kaggle/working/throttle-t4-validation/logs/kaggle-vllm-doctor.txt

kaggle-vllm doctor
Kaggle       : True
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
PyTorch path : /usr/local/lib/python3.12/dist-packages/torch/__init__.py
PyTorch CUDA : 12.8
CUDA usable  : True
NCCL         : 2.27.5
nvcc         : /usr/local/cuda/bin/nvcc
CUDA driver  : /usr/local/nvidia/lib64/libcuda.so
GPUs         : 2
  GPU 0: Tesla T4, compute 7.5
  GPU 1: Tesla T4, compute 7.5

Suggested build environment:
  export CUDA_HOME=/usr/local/cuda
  export CUDAToolkit_ROOT=/usr/local/cuda
  export VLLM_TARGET_DEVICE=cuda
  export TORCH_CUDA_ARCH_LIST=7.5
  export MAX_JOBS=1
  export NVCC_THREADS=1
  export CMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64

PASS: runtime matches the documented Kaggle T4x2 profile.


### Cell 14 — Verify native vLLM modules
Imports the staged vLLM runtime and native extensions (`vllm._C`, `vllm._moe_C`, `vllm.cumem_allocator`) while printing Python/PyTorch/CUDA/GPU properties. The recorded imports all pass.

In [22]:
%%bash
set -o pipefail

eval "$(kaggle-vllm env)"

python - <<'PY' | tee \
  /kaggle/working/throttle-t4-validation/logs/native-runtime.txt

import sys
import torch
import vllm

print("Python:", sys.version)
print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {p.name}, "
        f"CC={p.major}.{p.minor}, "
        f"VRAM={p.total_memory / 1024**2:.0f} MiB"
    )

for module in [
    "vllm._C",
    "vllm._moe_C",
    "vllm.cumem_allocator",
]:
    try:
        m = __import__(module, fromlist=["*"])
        print("PASS:", module, getattr(m, "__file__", None))
    except Exception as exc:
        print("FAIL:", module, repr(exc))
PY

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
vLLM: 0.18.2.dev0+ga26e8dc7f.d20260822
PyTorch: 2.10.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU count: 2
GPU 0: Tesla T4, CC=7.5, VRAM=14912 MiB
GPU 1: Tesla T4, CC=7.5, VRAM=14912 MiB
PASS: vllm._C /kaggle/working/vllm-staged/vllm/_C.abi3.so
PASS: vllm._moe_C /kaggle/working/vllm-staged/vllm/_moe_C.abi3.so
PASS: vllm.cumem_allocator /kaggle/working/vllm-staged/vllm/cumem_allocator.abi3.so


### Cell 15 — Download the TP=2 Qwen checkpoint
Downloads `waqasm86/kaggle-vllm-models` to `/kaggle/working/qwen2.5-3b-t4x2-sharded`. These artifacts are used as a vLLM `sharded_state` checkpoint for two tensor-parallel ranks.

In [23]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_DIR = "/kaggle/working/qwen2.5-3b-t4x2-sharded"

path = snapshot_download(
    repo_id="waqasm86/kaggle-vllm-models",
    local_dir=MODEL_DIR,
)

print("Downloaded to:", path)

for p in sorted(Path(MODEL_DIR).iterdir()):
    print(p.name)

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Downloaded to: /kaggle/working/qwen2.5-3b-t4x2-sharded
.cache
.gitattributes
LICENSE
NOTICE
README.md
SHARDED_STATE_SHA256SUMS.txt
config.json
generation_config.json
merges.txt
model-rank-0-part-0.safetensors
model-rank-0-part-1.safetensors
model-rank-1-part-0.safetensors
model-rank-1-part-1.safetensors
model.safetensors.index.json
tokenizer.json
tokenizer_config.json
vocab.json


### Cell 16 — Inspect the downloaded model
Shows the model directory size and files. The output confirms the approximately 5.8 GiB checkpoint, its rank-specific safetensor shards, tokenizer files, and configuration metadata.

In [24]:
%%bash

echo "===== MODEL DIRECTORY ====="
du -sh /kaggle/working/qwen2.5-3b-t4x2-sharded

echo
echo "===== FILES ====="
find /kaggle/working/qwen2.5-3b-t4x2-sharded \
  -maxdepth 2 \
  -type f \
  -printf '%10s  %p\n' | sort -n

===== MODEL DIRECTORY =====
5.8G	/kaggle/working/qwen2.5-3b-t4x2-sharded

===== FILES =====
       242  /kaggle/working/qwen2.5-3b-t4x2-sharded/generation_config.json
       392  /kaggle/working/qwen2.5-3b-t4x2-sharded/SHARDED_STATE_SHA256SUMS.txt
       513  /kaggle/working/qwen2.5-3b-t4x2-sharded/NOTICE
       661  /kaggle/working/qwen2.5-3b-t4x2-sharded/config.json
      1519  /kaggle/working/qwen2.5-3b-t4x2-sharded/.gitattributes
      3838  /kaggle/working/qwen2.5-3b-t4x2-sharded/README.md
      7305  /kaggle/working/qwen2.5-3b-t4x2-sharded/tokenizer_config.json
      7388  /kaggle/working/qwen2.5-3b-t4x2-sharded/LICENSE
     35581  /kaggle/working/qwen2.5-3b-t4x2-sharded/model.safetensors.index.json
   1671839  /kaggle/working/qwen2.5-3b-t4x2-sharded/merges.txt
   2776833  /kaggle/working/qwen2.5-3b-t4x2-sharded/vocab.json
   7031645  /kaggle/working/qwen2.5-3b-t4x2-sharded/tokenizer.json
 947544384  /kaggle/working/qwen2.5-3b-t4x2-sharded/model-rank-0-part-1.safetensors
 9475443

### Cell 17 — Check GPU state before server startup
Runs `nvidia-smi` before model launch to establish the pre-server GPU state.

In [25]:
!nvidia-smi

Thu Aug 27 05:05:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Cell 18 — Launch the dual-T4 OpenAI-compatible server
Starts `kaggle-vllm serve` in the background with Qwen, `sharded_state`, TP=2, max model length 2048, and 70% GPU-memory utilization. Server PID and logs are persisted.

In [26]:
%%bash
set -euxo pipefail

eval "$(kaggle-vllm env)"

MODEL_DIR="/kaggle/working/qwen2.5-3b-t4x2-sharded"
LOG="/kaggle/working/throttle-t4-validation/logs/kaggle-vllm-server.log"
PID="/kaggle/working/throttle-t4-validation/logs/kaggle-vllm-server.pid"

nohup kaggle-vllm serve "$MODEL_DIR" \
  --served-model-name qwen2.5-3b-kaggle-t4x2 \
  --load-format sharded_state \
  --tensor-parallel-size 2 \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.70 \
  > "$LOG" 2>&1 &

echo $! | tee "$PID"

echo
echo "Started server PID:"
cat "$PID"

360

Started server PID:
360


++ kaggle-vllm env
+ eval 'export PYTHONPATH=/kaggle/working/vllm-runtime-overlay:/kaggle/working/vllm-staged
export LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64
export PATH=/kaggle/working/vllm-staged/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin'
++ export PYTHONPATH=/kaggle/working/vllm-runtime-overlay:/kaggle/working/vllm-staged
++ PYTHONPATH=/kaggle/working/vllm-runtime-overlay:/kaggle/working/vllm-staged
++ export LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64
++ LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64
++ export PATH=/kaggle/working/vllm-staged/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/

### Cell 19 — Wait for server readiness
Polls `/v1/models` until the server responds. The observed output confirms the server is ready and exposes `qwen2.5-3b-kaggle-t4x2`.

In [27]:
%%bash

LOG="/kaggle/working/throttle-t4-validation/logs/kaggle-vllm-server.log"

for i in $(seq 1 150); do

    if curl -fsS \
        http://127.0.0.1:8000/v1/models \
        >/tmp/kaggle-vllm-models.json 2>/dev/null; then

        echo
        echo "========================"
        echo "SERVER READY"
        echo "========================"

        python -m json.tool /tmp/kaggle-vllm-models.json
        exit 0
    fi

    if (( i % 10 == 0 )); then
        echo
        echo "Still loading: attempt $i"
        echo "--- latest server output ---"
        tail -30 "$LOG"
    fi

    sleep 2
done

echo
echo "SERVER DID NOT BECOME READY"
tail -200 "$LOG"
exit 1


SERVER READY
{
    "object": "list",
    "data": [
        {
            "id": "qwen2.5-3b-kaggle-t4x2",
            "object": "model",
            "created": 1787807491,
            "owned_by": "vllm",
            "root": "/kaggle/working/qwen2.5-3b-t4x2-sharded",
            "parent": null,
            "max_model_len": 2048,
            "permission": [
                {
                    "id": "modelperm-a738b6ad20d11d80",
                    "object": "model_permission",
                    "created": 1787807491,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


### Cell 20 — Inspect GPU allocation after model load
Runs `nvidia-smi` again after server startup so model-serving memory allocation across both T4s is visible.

In [28]:
!nvidia-smi

Thu Aug 27 05:11:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             27W /   70W |   10857MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Cell 21 — Save GPU-process / TP-worker evidence
Writes `nvidia-smi` and compute-process information to `logs/server-gpu-state.txt`, preserving direct evidence of vLLM workers on both GPUs.

In [29]:
%%bash

{
    date -u

    echo
    echo "===== NVIDIA-SMI ====="
    nvidia-smi

    echo
    echo "===== COMPUTE PROCESSES ====="
    nvidia-smi \
      --query-compute-apps=gpu_uuid,pid,process_name,used_memory \
      --format=csv

} | tee \
  /kaggle/working/throttle-t4-validation/logs/server-gpu-state.txt

Thu Aug 27 05:11:56 AM UTC 2026

===== NVIDIA-SMI =====
Thu Aug 27 05:11:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             27W /   70W |   10857MiB /  15360MiB |      0%      Default |
|                                         |                        |                

### Cell 22 — Extract distributed-runtime evidence
Filters the server log for NCCL, rank, tensor-parallel, attention-backend, and T4/SM75 details. The output shows world size 2, NCCL 2.27.5, TP ranks 0/1, the expected unsupported SymmMem warning on CC 7.5, and `TRITON_ATTN`.

In [30]:
%%bash

LOG=/kaggle/working/throttle-t4-validation/logs/kaggle-vllm-server.log

grep -Ei \
'nccl|tensor.parallel|tensor_parallel|rank|triton|attention|symm|Tesla T4|cuda' \
"$LOG" | tail -200

(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:233] non-default args: {'model_tag': '/kaggle/working/qwen2.5-3b-t4x2-sharded', 'host': '127.0.0.1', 'model': '/kaggle/working/qwen2.5-3b-t4x2-sharded', 'dtype': 'float16', 'max_model_len': 2048, 'enforce_eager': True, 'served_model_name': ['qwen2.5-3b-kaggle-t4x2'], 'load_format': 'sharded_state', 'tensor_parallel_size': 2, 'disable_custom_all_reduce': True, 'gpu_memory_utilization': 0.7}
(APIServer pid=370) WARNING 08-27 05:06:23 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(APIServer pid=370) INFO 08-27 05:06:23 [vllm.py:985] Cudagraph is disabled under eager mode
(EngineCore pid=418) INFO 08-27 05:06:39 [core.py:103] Initializing a V1 LLM engine (v0.18.2.dev0+ga26e8dc7f.d20260822) with config: model='/kaggle/working/qwen2.5-3b-t4x2-sharded', speculative_config=None, tokenizer='/kaggle/working/qwen2.5-3b-t4x2-sharded', skip_tokenizer_init=

### Cell 23 — Show the recent complete vLLM server log
Prints the last 200 server-log lines to keep initialization/runtime evidence directly in the executed notebook.

In [31]:
!tail -200 /kaggle/working/throttle-t4-validation/logs/kaggle-vllm-server.log

(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:297] 
(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:297]        █     █     █▄   ▄█
(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:297]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.18.2.dev0+ga26e8dc7f.d20260822
(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:297]   █▄█▀ █     █     █     █  model   /kaggle/working/qwen2.5-3b-t4x2-sharded
(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:297]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:297] 
(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:233] non-default args: {'model_tag': '/kaggle/working/qwen2.5-3b-t4x2-sharded', 'host': '127.0.0.1', 'model': '/kaggle/working/qwen2.5-3b-t4x2-sharded', 'dtype': 'float16', 'max_model_len': 2048, 'enforce_eager': True, 'served_model_name': ['qwen2.5-3b-kaggle-t4x2'], 'load_format': 'sharded_state', 'tensor_parallel_size': 2, 'disable_custom_all_reduce': True, 'gpu_memory_utilization': 0.7}
(APIServer pid=370) INFO 0

### Cell 24 — Verify `/v1/models`
Calls the OpenAI-compatible model-list endpoint directly and pretty-prints the response before Throttle sends any workload.

In [32]:
!curl -sS http://127.0.0.1:8000/v1/models | python -m json.tool

{
    "object": "list",
    "data": [
        {
            "id": "qwen2.5-3b-kaggle-t4x2",
            "object": "model",
            "created": 1787807744,
            "owned_by": "vllm",
            "root": "/kaggle/working/qwen2.5-3b-t4x2-sharded",
            "parent": null,
            "max_model_len": 2048,
            "permission": [
                {
                    "id": "modelperm-b54149a9d05a1b8c",
                    "object": "model_permission",
                    "created": 1787807744,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


### Cell 25 — Verify end-to-end chat inference
Sends a deterministic chat request to `/v1/chat/completions` and saves the JSON response. The observed model response is `Kaggle dual T4 ready`.

In [33]:
%%bash

curl -sS \
  http://127.0.0.1:8000/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{
    "model": "qwen2.5-3b-kaggle-t4x2",
    "messages": [
      {
        "role": "user",
        "content": "Reply with exactly: Kaggle dual T4 ready"
      }
    ],
    "temperature": 0,
    "max_tokens": 20
  }' \
  | tee \
  /kaggle/working/throttle-t4-validation/results/manual-chat.json \
  | python -m json.tool

{
    "id": "chatcmpl-943cc268ce14f9b6",
    "object": "chat.completion",
    "created": 1787807757,
    "model": "qwen2.5-3b-kaggle-t4x2",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "Kaggle dual T4 ready",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [],
                "reasoning": null
            },
            "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
            "token_ids": null
        }
    ],
    "service_tier": null,
    "system_fingerprint": null,
    "usage": {
        "prompt_tokens": 39,
        "total_tokens": 47,
        "completion_tokens": 8,
        "prompt_tokens_details": null
    },
    "prompt_logprobs": null,
    "prompt_token_ids": null,
    "kv_transfer_params": null
}


### Cell 26 — Capture the full Prometheus baseline
Downloads `/metrics` to a completed file first, records HTTP metadata, and only then displays its first 100 lines. The observed response is HTTP 200 with 53,927 bytes, avoiding the earlier `curl | tee | head` SIGPIPE problem.

In [39]:
%%bash
set -euo pipefail

RESULTS=/kaggle/working/throttle-t4-validation/results
mkdir -p "$RESULTS"

METRICS="$RESULTS/metrics-baseline.txt"
HEADERS="$RESULTS/metrics-baseline-headers.txt"
SUMMARY="$RESULTS/metrics-baseline-http.txt"

rm -f "$METRICS" "$HEADERS" "$SUMMARY"

curl \
  --fail \
  --silent \
  --show-error \
  --dump-header "$HEADERS" \
  --output "$METRICS" \
  --write-out 'http_code=%{http_code}\nsize_download=%{size_download}\ntime_total=%{time_total}\n' \
  http://127.0.0.1:8000/metrics \
  | tee "$SUMMARY"

echo
echo "===== METRICS FILE ====="
ls -lh "$METRICS"

echo
echo "lines=$(wc -l < "$METRICS")"
echo "bytes=$(wc -c < "$METRICS")"

echo
echo "===== FIRST 100 LINES ====="
sed -n '1,100p' "$METRICS"

http_code=200
size_download=53927
time_total=0.006518

===== METRICS FILE =====
-rw-r--r-- 1 root root 53K Aug 27 05:31 /kaggle/working/throttle-t4-validation/results/metrics-baseline.txt

lines=594
bytes=53927

===== FIRST 100 LINES =====
# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 63950.0
python_gc_objects_collected_total{generation="1"} 7547.0
python_gc_objects_collected_total{generation="2"} 562.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generatio

### Cell 27 — Extract vLLM-specific Prometheus samples
Filters the full baseline for vLLM HELP/TYPE/sample lines and writes a focused metrics file. The output reports 483 vLLM-relevant lines.

In [40]:
%%bash
set -euo pipefail

RESULTS=/kaggle/working/throttle-t4-validation/results
METRICS="$RESULTS/metrics-baseline.txt"
OUT="$RESULTS/metrics-vllm-baseline-relevant.txt"

grep -E \
'^# (HELP|TYPE) vllm|^vllm:' \
"$METRICS" > "$OUT" || true

echo "Relevant vLLM metric lines:"
wc -l "$OUT"

echo
sed -n '1,250p' "$OUT"

Relevant vLLM metric lines:
483 /kaggle/working/throttle-t4-validation/results/metrics-vllm-baseline-relevant.txt

# HELP vllm:estimated_flops_per_gpu_total Estimated number of floating point operations per GPU (for Model Flops Utilization calculations).
# TYPE vllm:estimated_flops_per_gpu_total counter
vllm:estimated_flops_per_gpu_total{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
# HELP vllm:estimated_flops_per_gpu_created Estimated number of floating point operations per GPU (for Model Flops Utilization calculations).
# TYPE vllm:estimated_flops_per_gpu_created gauge
vllm:estimated_flops_per_gpu_created{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 1.787807239096318e+09
# HELP vllm:estimated_read_bytes_per_gpu_total Estimated number of bytes read from memory per GPU (for Model Flops Utilization calculations).
# TYPE vllm:estimated_read_bytes_per_gpu_total counter
vllm:estimated_read_bytes_per_gpu_total{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
# HELP vllm:estimate

### Cell 28 — Inventory unique vLLM metric names
Extracts and sorts names from `# HELP vllm:` lines. The run records 67 unique vLLM metric names.

In [41]:
%%bash
set -euo pipefail

RESULTS=/kaggle/working/throttle-t4-validation/results

grep '^# HELP vllm:' \
  "$RESULTS/metrics-baseline.txt" \
  | awk '{print $3}' \
  | sort -u \
  > "$RESULTS/vllm-prometheus-metric-names.txt"

echo "Unique vLLM metrics:"
wc -l "$RESULTS/vllm-prometheus-metric-names.txt"

cat "$RESULTS/vllm-prometheus-metric-names.txt"

Unique vLLM metrics:
67 /kaggle/working/throttle-t4-validation/results/vllm-prometheus-metric-names.txt
vllm:cache_config_info
vllm:e2e_request_latency_seconds
vllm:e2e_request_latency_seconds_created
vllm:engine_sleep_state
vllm:estimated_flops_per_gpu_created
vllm:estimated_flops_per_gpu_total
vllm:estimated_read_bytes_per_gpu_created
vllm:estimated_read_bytes_per_gpu_total
vllm:estimated_write_bytes_per_gpu_created
vllm:estimated_write_bytes_per_gpu_total
vllm:external_prefix_cache_hits_created
vllm:external_prefix_cache_hits_total
vllm:external_prefix_cache_queries_created
vllm:external_prefix_cache_queries_total
vllm:generation_tokens_created
vllm:generation_tokens_total
vllm:inter_token_latency_seconds
vllm:inter_token_latency_seconds_created
vllm:iteration_tokens_total
vllm:iteration_tokens_total_created
vllm:kv_cache_usage_perc
vllm:mm_cache_hits_created
vllm:mm_cache_hits_total
vllm:mm_cache_queries_created
vllm:mm_cache_queries_total
vllm:num_preemptions_created
vllm:num_pree

### Cell 29 — Verify all server endpoints before Throttle
Checks `/health`, `/v1/models`, and `/metrics`, saving the status/results. All three endpoints are shown as healthy before benchmarking begins.

In [42]:
%%bash
set -euo pipefail

RESULTS=/kaggle/working/throttle-t4-validation/results

{
    echo "===== UTC ====="
    date -u

    echo
    echo "===== HEALTH ====="
    curl -fsS -o /dev/null \
      -w 'HTTP %{http_code}\n' \
      http://127.0.0.1:8000/health

    echo
    echo "===== MODELS ====="
    curl -fsS http://127.0.0.1:8000/v1/models

    echo
    echo "===== METRICS ====="
    curl -fsS -o /dev/null \
      -w 'HTTP %{http_code}\n' \
      http://127.0.0.1:8000/metrics

} | tee "$RESULTS/server-endpoint-health.txt"

===== UTC =====
Thu Aug 27 05:34:39 AM UTC 2026

===== HEALTH =====
HTTP 200

===== MODELS =====
{"object":"list","data":[{"id":"qwen2.5-3b-kaggle-t4x2","object":"model","created":1787808879,"owned_by":"vllm","root":"/kaggle/working/qwen2.5-3b-t4x2-sharded","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-8ff2a943411253e9","object":"model_permission","created":1787808879,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}
===== METRICS =====
HTTP 200


### Cell 30 — Capture live TP=2 process state
Records GPU capability/memory/utilization, compute processes, and vLLM processes immediately before Throttle testing.

In [43]:
%%bash
set -euo pipefail

LOGS=/kaggle/working/throttle-t4-validation/logs

{
    echo "===== UTC ====="
    date -u

    echo
    echo "===== GPU SUMMARY ====="
    nvidia-smi \
      --query-gpu=index,name,compute_cap,memory.total,memory.used,utilization.gpu,driver_version \
      --format=csv

    echo
    echo "===== COMPUTE PROCESSES ====="
    nvidia-smi \
      --query-compute-apps=gpu_uuid,pid,process_name,used_memory \
      --format=csv

    echo
    echo "===== VLLM PROCESSES ====="
    ps -ef | grep -E '[v]llm|[k]aggle-vllm' || true

} | tee "$LOGS/live-server-state.txt"

===== UTC =====
Thu Aug 27 05:34:51 AM UTC 2026

===== GPU SUMMARY =====
index, name, compute_cap, memory.total [MiB], memory.used [MiB], utilization.gpu [%], driver_version
0, Tesla T4, 7.5, 15360 MiB, 10857 MiB, 0 %, 580.159.04
1, Tesla T4, 7.5, 15360 MiB, 10857 MiB, 0 %, 580.159.04

===== COMPUTE PROCESSES =====
gpu_uuid, pid, process_name, used_gpu_memory [MiB]
GPU-07e7c9bf-96bc-2b52-b9fa-74b6230b884f, 442, VLLM::Worker_TP0, 10854 MiB
GPU-3ee03875-89de-6687-feec-5bb593e00ec8, 443, VLLM::Worker_TP1, 10854 MiB

===== VLLM PROCESSES =====
root         360       1  0 05:05 ?        00:00:02 /usr/bin/python3 /usr/local/bin/kaggle-vllm serve /kaggle/working/qwen2.5-3b-t4x2-sharded --served-model-name qwen2.5-3b-kaggle-t4x2 --load-format sharded_state --tensor-parallel-size 2 --max-model-len 2048 --gpu-memory-utilization 0.70
root         370     360  1 05:05 ?        00:00:34 /usr/bin/python3 /kaggle/working/vllm-staged/bin/vllm serve /kaggle/working/qwen2.5-3b-t4x2-sharded --served-mode

### Cell 31 — Freeze a pre-Throttle baseline
Copies the key environment, server, inference, metrics, and GPU evidence into `pre-throttle-baseline/` and calculates SHA-256 checksums.

In [44]:
%%bash
set -euo pipefail

ROOT=/kaggle/working/throttle-t4-validation
SNAP="$ROOT/pre-throttle-baseline"

rm -rf "$SNAP"
mkdir -p "$SNAP"

cp "$ROOT/logs/environment.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/logs/kaggle-vllm-doctor.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/logs/native-runtime.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/logs/server-gpu-state.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/logs/live-server-state.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/logs/kaggle-vllm-server.log" "$SNAP/" 2>/dev/null || true

cp "$ROOT/results/manual-chat.json" "$SNAP/" 2>/dev/null || true
cp "$ROOT/results/metrics-baseline.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/results/metrics-baseline-headers.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/results/metrics-baseline-http.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/results/vllm-prometheus-metric-names.txt" "$SNAP/" 2>/dev/null || true
cp "$ROOT/results/server-endpoint-health.txt" "$SNAP/" 2>/dev/null || true

find "$SNAP" -maxdepth 1 -type f \
  -exec sha256sum {} \; \
  > "$SNAP/SHA256SUMS.txt"

echo "===== PRE-THROTTLE SNAPSHOT ====="
ls -lh "$SNAP"

echo
cat "$SNAP/SHA256SUMS.txt"

===== PRE-THROTTLE SNAPSHOT =====
total 116K
-rw-r--r-- 1 root root 3.3K Aug 27 05:35 environment.txt
-rw-r--r-- 1 root root  742 Aug 27 05:35 kaggle-vllm-doctor.txt
-rw-r--r-- 1 root root  13K Aug 27 05:35 kaggle-vllm-server.log
-rw-r--r-- 1 root root 1.3K Aug 27 05:35 live-server-state.txt
-rw-r--r-- 1 root root  596 Aug 27 05:35 manual-chat.json
-rw-r--r-- 1 root root  152 Aug 27 05:35 metrics-baseline-headers.txt
-rw-r--r-- 1 root root   54 Aug 27 05:35 metrics-baseline-http.txt
-rw-r--r-- 1 root root  53K Aug 27 05:35 metrics-baseline.txt
-rw-r--r-- 1 root root  459 Aug 27 05:35 native-runtime.txt
-rw-r--r-- 1 root root  622 Aug 27 05:35 server-endpoint-health.txt
-rw-r--r-- 1 root root 2.4K Aug 27 05:35 server-gpu-state.txt
-rw-r--r-- 1 root root 1.9K Aug 27 05:35 SHA256SUMS.txt
-rw-r--r-- 1 root root 2.3K Aug 27 05:35 vllm-prometheus-metric-names.txt

517251bf7c77d98916c4fb61d77946e6ae480502179235e02a23cba1134068a3  /kaggle/working/throttle-t4-validation/pre-throttle-baseline/ka

### Cell 32 — Reconfirm the live backend
Checks all three endpoints and both TP workers again immediately before running Throttle workload tests.

In [47]:
%%bash
set -euo pipefail

echo "===== SERVER CHECK ====="

curl -fsS \
  -o /dev/null \
  -w 'health_http=%{http_code}\n' \
  http://127.0.0.1:8000/health

curl -fsS \
  -o /dev/null \
  -w 'models_http=%{http_code}\n' \
  http://127.0.0.1:8000/v1/models

curl -fsS \
  -o /dev/null \
  -w 'metrics_http=%{http_code}\n' \
  http://127.0.0.1:8000/metrics

echo
nvidia-smi \
  --query-compute-apps=pid,process_name,used_memory \
  --format=csv

===== SERVER CHECK =====
health_http=200
models_http=200
metrics_http=200

pid, process_name, used_gpu_memory [MiB]
442, VLLM::Worker_TP0, 10854 MiB
443, VLLM::Worker_TP1, 10854 MiB


### Cell 33 — Record Throttle source state at test time
Captures UTC time, Throttle 0.3.0, exact commit `7c470078...`, and the Git working-tree status before benchmark traffic.

In [48]:
%%bash
set -euo pipefail

ROOT=/kaggle/working/throttle-t4-validation
cd /kaggle/working/throttle

{
    echo "===== TEST TIMESTAMP ====="
    date -u

    echo
    echo "===== THROTTLE VERSION ====="
    throttle --version

    echo
    echo "===== GIT COMMIT ====="
    git rev-parse HEAD

    echo
    echo "===== GIT STATUS ====="
    git status --short

} | tee "$ROOT/logs/throttle-pre-test-state.txt"

===== TEST TIMESTAMP =====
Thu Aug 27 05:54:36 AM UTC 2026

===== THROTTLE VERSION =====
throttle 0.3.0

===== GIT COMMIT =====
7c470078de78f4771b190db366be391e9f8b6e44

===== GIT STATUS =====


### Cell 34 — Display `throttle smoke` help
Shows the exact arguments supported by the installed smoke command so the executed test can be matched to the tested CLI.

In [49]:
!throttle smoke --help

usage: throttle smoke [-h] --model MODEL --url URL [--api-key-env NAME]
                      [--allow-insecure-http] [--backend {native,guidellm}]
                      [--guidellm-prompt-tokens GUIDELLM_PROMPT_TOKENS]
                      [--guidellm-executable GUIDELLM_EXECUTABLE]
                      [--allow-guidellm-validation-gaps]
                      [--cost-model {unknown,dedicated-hourly,serverless-active-seconds,user-supplied}]
                      [--gpus GPUS]
                      [--total-hourly-price TOTAL_HOURLY_PRICE | --per-gpu-hourly-price PER_GPU_HOURLY_PRICE]
                      [--active-second-price ACTIVE_SECOND_PRICE]
                      [--max-active-workers MAX_ACTIVE_WORKERS]
                      [--billed-active-seconds BILLED_ACTIVE_SECONDS]
                      [--user-supplied-total USER_SUPPLIED_TOTAL]
                      [--allow-unknown-cost] [--prompts PROMPTS]
                      [--warmup-prompts WARMUP_PROMPTS]
                    

### Cell 35 — Run Throttle smoke against the dual-T4 backend
Executes the smoke path against the local OpenAI-compatible endpoint, persisting log and JSON artifacts. The captured output shows successful smoke measurements at concurrency 1, 4, and 8.

In [50]:
%%bash
set -o pipefail

ROOT=/kaggle/working/throttle-t4-validation
RESULT="$ROOT/results/throttle-smoke-t4x2.json"
LOG="$ROOT/logs/throttle-smoke-t4x2.log"

rm -f "$RESULT"

export VLLM_API_KEY="kaggle-local-validation"

throttle smoke \
  --model qwen2.5-3b-kaggle-t4x2 \
  --url http://127.0.0.1:8000/v1 \
  --api-key-env VLLM_API_KEY \
  --cost-model unknown \
  --allow-unknown-cost \
  --output "$RESULT" \
  2>&1 | tee "$LOG"

STATUS=${PIPESTATUS[0]}

echo
echo "THROTTLE_SMOKE_EXIT_CODE=$STATUS"

exit "$STATUS"

Throttle SMOKE — SHORT SAMPLE, NON-DECISION-GRADE
 condition              valid  grade   requests  block-mean tok/s   p95 e2e   p95 TTFT  SLO goodput
 closed_loop:1            yes     no     8/8                15.65    8334.67     122.94            -
   evidence note: smoke_mode_is_not_decision_grade
 closed_loop:4            yes     no     8/8                50.40   11141.26    1447.92            -
   evidence note: smoke_mode_is_not_decision_grade
 closed_loop:8            yes     no     8/8               120.83    8139.91     116.43            -
   evidence note: smoke_mode_is_not_decision_grade
Best Tested Concurrency: 8 (not_applicable_smoke; boundary reached=true).
Claim boundary: descriptive smoke observation only; never a production recommendation.
Cost model: unknown; total=-; $/1M output=-.
Measurements describe only this exact workload and declared manifest. They are not a universal optimization, production recommendation, or savings claim.
Sanitized JSON report: /kaggle/wor

### Cell 36 — Inspect the structured smoke report
Loads and prints `throttle-smoke-t4x2.json`, preserving the structured report in the notebook as well as on disk.

In [51]:
import json
from pathlib import Path

p = Path(
    "/kaggle/working/throttle-t4-validation/"
    "results/throttle-smoke-t4x2.json"
)

if p.exists():
    report = json.loads(p.read_text())

    print("Smoke report created:", p)
    print()
    print(json.dumps(report, indent=2)[:20000])
else:
    print("No smoke JSON was created. Review throttle-smoke-t4x2.log.")

Smoke report created: /kaggle/working/throttle-t4-validation/results/throttle-smoke-t4x2.json

{
  "artifact_type": "throttle_run",
  "best_tested": {
    "available": true,
    "block_mean_output_tokens_per_second": 120.82743373689298,
    "block_mean_output_tokens_per_second_ci": {
      "confidence": 0.95,
      "high": null,
      "low": null,
      "method": "student_t_blocks",
      "n": 1
    },
    "boundary_reached": true,
    "claim": "descriptive smoke observation only; never a production recommendation",
    "completion_tokens_per_response_relative_spread": 0.0,
    "completion_tokens_per_response_tolerance": 0.05,
    "condition_id": "closed_loop:8",
    "field": "best_tested_concurrency",
    "optimum_found": false,
    "pooled_output_tokens_per_second": 120.82743373689298,
    "reasons": [
      "smoke_sample_is_not_decision_grade",
      "throughput_confidence_intervals_overlap"
    ],
    "state": "not_applicable_smoke",
    "value": 8
  },
  "completed_at": "2026-08-2

### Cell 37 — Capture Prometheus state after smoke
Downloads `/metrics` again after Throttle traffic and displays selected request/token/cache metrics to demonstrate that the exporter remains available and reflects workload activity.

In [52]:
%%bash
set -euo pipefail

RESULTS=/kaggle/working/throttle-t4-validation/results

curl \
  --fail \
  --silent \
  --show-error \
  --output "$RESULTS/metrics-after-smoke.txt" \
  http://127.0.0.1:8000/metrics

echo "bytes=$(wc -c < "$RESULTS/metrics-after-smoke.txt")"
echo "lines=$(wc -l < "$RESULTS/metrics-after-smoke.txt")"

grep -E \
'^vllm:(prompt_tokens_total|generation_tokens_total|request_success_total|num_requests_running|num_requests_waiting|kv_cache_usage_perc)' \
"$RESULTS/metrics-after-smoke.txt" || true

bytes=54221
lines=594
vllm:num_requests_running{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
vllm:num_requests_waiting{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
vllm:kv_cache_usage_perc{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
vllm:prompt_tokens_total{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 1234.0
vllm:generation_tokens_total{engine="0",model_name="qwen2.5-3b-kaggle-t4x2"} 2975.0
vllm:request_success_total{engine="0",finished_reason="stop",model_name="qwen2.5-3b-kaggle-t4x2"} 7.0
vllm:request_success_total{engine="0",finished_reason="length",model_name="qwen2.5-3b-kaggle-t4x2"} 21.0
vllm:request_success_total{engine="0",finished_reason="abort",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
vllm:request_success_total{engine="0",finished_reason="error",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0
vllm:request_success_total{engine="0",finished_reason="repetition",model_name="qwen2.5-3b-kaggle-t4x2"} 0.0


### Cell 38 — Inspect Throttle's chat URL construction
Searches `src/throttle/cli.py` around all `chat/completions` occurrences and saves the source context before `validate-sim` is run.

In [53]:
%%bash

cd /kaggle/working/throttle

echo "===== validate-sim related URLs in source ====="

grep -n \
  -B 4 \
  -A 4 \
  'chat/completions' \
  src/throttle/cli.py \
  | tee \
  /kaggle/working/throttle-t4-validation/logs/validate-sim-url-source.txt

===== validate-sim related URLs in source =====
2530-                prompt = "Test " * prompt_tokens
2531-
2532-                req_start = time.time()
2533-                response = client.post(
2534:                    f"{args.endpoint_url}/chat/completions",
2535-                    headers=headers,
2536-                    json={
2537-                        "model": args.model,
2538-                        "messages": [{"role": "user", "content": prompt}],
--
2983-    print(f"Testing connection to {args.endpoint_url}...")
2984-    try:
2985-        with httpx.Client(timeout=10.0) as client:
2986-            response = client.post(
2987:                f"{args.endpoint_url}/v1/chat/completions",
2988-                headers=headers,
2989-                json={
2990-                    "model": args.model,
2991-                    "messages": [{"role": "user", "content": "test"}],
--
3059-
3060-                async with httpx.AsyncClient(timeout=120.0) as client:
3061-           

### Cell 39 — Run unmodified `validate-sim`
Executes Throttle's simulator validation against a base URL ending in `/v1`. The `$1.00/hour` value is a synthetic normalization input required by the CLI, not Kaggle pricing. The unmodified command connects successfully but then fails during Light load with HTTP 404; the exit code is preserved while the notebook continues.

In [54]:
%%bash
set +e
set -o pipefail

ROOT=/kaggle/working/throttle-t4-validation
LOG="$ROOT/logs/validate-sim-unmodified-t4x2.log"

cd /kaggle/working/throttle

throttle validate-sim \
  --endpoint-url http://127.0.0.1:8000/v1 \
  --model qwen2.5-3b-kaggle-t4x2 \
  --gpu-hourly-rate 1.00 \
  2>&1 | tee "$LOG"

STATUS=${PIPESTATUS[0]}

echo
echo "VALIDATE_SIM_EXIT_CODE=$STATUS"

printf '%s\n' "$STATUS" > \
  "$ROOT/results/validate-sim-exit-code.txt"

# Intentionally do not fail the Kaggle notebook.
exit 0

Throttle Simulator Validation

Testing connection to http://127.0.0.1:8000/v1...
Connection successful.

Running Light load (1 req/sec, 20 requests)...

Error: Request failed: Request 1 failed with status 404

VALIDATE_SIM_EXIT_CODE=1


### Cell 40 — Preserve the original 404 evidence
Copies the unmodified simulator log and exit code to dedicated evidence files before changing any Throttle source.

In [57]:
%%bash
set -euo pipefail

ROOT=/kaggle/working/throttle-t4-validation

cp "$ROOT/logs/validate-sim-unmodified-t4x2.log" \
   "$ROOT/logs/validate-sim-original-404-evidence.log"

cp "$ROOT/results/validate-sim-exit-code.txt" \
   "$ROOT/results/validate-sim-original-exit-code.txt"

echo "Original failure preserved."

Original failure preserved.


### Cell 41 — Apply a local test-only URL patch
Replaces three local occurrences of `f"{args.endpoint_url}/v1/chat/completions"` with `f"{args.endpoint_url}/chat/completions"`. This is an experiment to continue validation after the 404, not a claim that this broad replacement is the final upstream fix.

In [58]:
from pathlib import Path

p = Path("/kaggle/working/throttle/src/throttle/cli.py")
text = p.read_text()

old = 'f"{args.endpoint_url}/v1/chat/completions"'
new = 'f"{args.endpoint_url}/chat/completions"'

count = text.count(old)

print("Occurrences before patch:", count)

if count == 0:
    raise RuntimeError("Expected validate-sim URL pattern not found.")

text = text.replace(old, new)

p.write_text(text)

print("Patched.")
print("Remaining old occurrences:", p.read_text().count(old))

Occurrences before patch: 3
Patched.
Remaining old occurrences: 0


### Cell 42 — Save the exact source diff
Captures `git diff -- src/throttle/cli.py` into `validate-sim-local-fix.diff` so the test-only modification is fully reviewable.

In [59]:
%%bash
set -euo pipefail

cd /kaggle/working/throttle

git diff -- src/throttle/cli.py \
  | tee \
  /kaggle/working/throttle-t4-validation/logs/validate-sim-local-fix.diff

diff --git a/src/throttle/cli.py b/src/throttle/cli.py
index 67644b0..37f8e69 100644
--- a/src/throttle/cli.py
+++ b/src/throttle/cli.py
@@ -2984,7 +2984,7 @@ def _handle_measure(args: argparse.Namespace) -> int:
     try:
         with httpx.Client(timeout=10.0) as client:
             response = client.post(
-                f"{args.endpoint_url}/v1/chat/completions",
+                f"{args.endpoint_url}/chat/completions",
                 headers=headers,
                 json={
                     "model": args.model,
@@ -3060,7 +3060,7 @@ def _handle_measure(args: argparse.Namespace) -> int:
                 async with httpx.AsyncClient(timeout=120.0) as client:
                     try:
                         response = await client.post(
-                            f"{args.endpoint_url}/v1/chat/completions",
+                            f"{args.endpoint_url}/chat/completions",
                             headers=headers,
                             json={
               

### Cell 43 — Reinstall the local patched source
Force-reinstalls the modified Throttle checkout without dependencies and verifies the CLI remains version 0.3.0.

In [60]:
%%bash
set -euo pipefail

cd /kaggle/working/throttle

python -m pip install \
  --no-deps \
  --force-reinstall \
  .

throttle --version

Processing /kaggle/working/throttle
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for throttle-pro: filename=throttle_pro-0.3.0-py3-none-any.whl size=188931 sha256=e6be925e23ed332f1e5831f6873bb7e3e8d3f064a2f3263ffaa23e9c9e415160
  Stored in directory: /tmp/pip-ephem-wheel-cache-v0pcg86d/wheels/23/0d/34/e09155650ca477ff9ae51de1d93fec1764c0aed5d77537ba4f
Successfully built throttle-pro
  Attempting uninstall: throttle-pro
    Found existing installation: throttle-pro 0.3.0
    Uninstalling throttle-pro-0.3.0:
      Successfully uninstalled throttle-pro-0.3.0
throttle 0.3.0


### Cell 44 — Reconfirm backend health after reinstall
Checks health, models, metrics, and both TP workers after the local package reinstall. All remain healthy.

In [61]:
%%bash
set -euo pipefail

curl -fsS \
  -o /dev/null \
  -w 'health=%{http_code}\n' \
  http://127.0.0.1:8000/health

curl -fsS \
  -o /dev/null \
  -w 'models=%{http_code}\n' \
  http://127.0.0.1:8000/v1/models

curl -fsS \
  -o /dev/null \
  -w 'metrics=%{http_code}\n' \
  http://127.0.0.1:8000/metrics

nvidia-smi \
  --query-compute-apps=process_name,used_memory \
  --format=csv

health=200
models=200
metrics=200
process_name, used_gpu_memory [MiB]
VLLM::Worker_TP0, 10854 MiB
VLLM::Worker_TP1, 10854 MiB


### Cell 45 — Rerun simulator validation with the local URL patch
Runs the patched validation and preserves its full log and exit code. The patched run reaches real Light and Medium measurements, then Heavy fails at request 86 with HTTP 400. The captured output also includes an AnyIO/httpcore shutdown `ValueError`; the notebook intentionally preserves these failures rather than masking them.

In [62]:
%%bash
set +e
set -o pipefail

ROOT=/kaggle/working/throttle-t4-validation
LOG="$ROOT/logs/validate-sim-patched-t4x2.log"

cd /kaggle/working/throttle

throttle validate-sim \
  --endpoint-url http://127.0.0.1:8000/v1 \
  --model qwen2.5-3b-kaggle-t4x2 \
  --gpu-hourly-rate 1.00 \
  2>&1 | tee "$LOG"

STATUS=${PIPESTATUS[0]}

echo
echo "VALIDATE_SIM_PATCHED_EXIT_CODE=$STATUS"

printf '%s\n' "$STATUS" > \
  "$ROOT/results/validate-sim-patched-exit-code.txt"

exit 0

unhandled exception during asyncio.run() shutdown
task: <Task finished name='Task-242' coro=<_handle_validate_sim.<locals>.run_concurrent_workload.<locals>.send_request() done, defined at /usr/local/lib/python3.12/dist-packages/throttle/cli.py:3336> exception=ValueError('second argument (exceptions) must be a non-empty sequence')>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/throttle/cli.py", line 3354, in send_request
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpx/_client.py", line 1859, in post
    return await self.request(
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpx/_client.py", line 1540, in request
    return await self.send(request, auth=auth, follow_redirects=follow_redirects)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpx/_client.py", lin

### Cell 46 — Capture server context for the Heavy-load 400
Searches the vLLM log for HTTP 400 and related error/context-length clues, then appends the final 200 server-log lines. No further fix is attempted in this notebook.

In [65]:
%%bash
set -euo pipefail

ROOT=/kaggle/working/throttle-t4-validation
LOG="$ROOT/logs/kaggle-vllm-server.log"
OUT="$ROOT/logs/validate-sim-heavy-400-server-context.txt"

{
    echo "===== HTTP 400 / ERROR CONTEXT ====="

    grep -nEi \
      '400|bad request|maximum context|max.*len|prompt.*too|input.*too|error|exception' \
      "$LOG" \
      | tail -100

    echo
    echo "===== FINAL 200 SERVER LOG LINES ====="
    tail -200 "$LOG"

} > "$OUT"

cat "$OUT"

===== HTTP 400 / ERROR CONTEXT =====
7:(APIServer pid=370) INFO 08-27 05:06:06 [utils.py:233] non-default args: {'model_tag': '/kaggle/working/qwen2.5-3b-t4x2-sharded', 'host': '127.0.0.1', 'model': '/kaggle/working/qwen2.5-3b-t4x2-sharded', 'dtype': 'float16', 'max_model_len': 2048, 'enforce_eager': True, 'served_model_name': ['qwen2.5-3b-kaggle-t4x2'], 'load_format': 'sharded_state', 'tensor_parallel_size': 2, 'disable_custom_all_reduce': True, 'gpu_memory_utilization': 0.7}
10:(APIServer pid=370) INFO 08-27 05:06:22 [model.py:1582] Using max model len 2048
17:(EngineCore pid=418) INFO 08-27 05:06:39 [core.py:103] Initializing a V1 LLM engine (v0.18.2.dev0+ga26e8dc7f.d20260822) with config: model='/kaggle/working/qwen2.5-3b-t4x2-sharded', speculative_config=None, tokenizer='/kaggle/working/qwen2.5-3b-t4x2-sharded', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, 

### Cell 47 — Generate the human-readable validation summary
Writes `VALIDATION-SUMMARY.md` with the environment, baseline checks, smoke results, original URL 404, patched Light/Medium simulator errors, Heavy HTTP 400, async shutdown issue, and interpretation.

In [67]:
%%bash
set -euo pipefail

ROOT=/kaggle/working/throttle-t4-validation
OUT="$ROOT/VALIDATION-SUMMARY.md"

cat > "$OUT" <<'EOF'
# Throttle validation on Kaggle dual NVIDIA Tesla T4

## Environment

- Throttle: 0.3.0
- kaggle-vllm: 0.1.2
- Backend: upstream-derived vLLM runtime
- GPU: 2 x NVIDIA Tesla T4
- Compute capability: 7.5 / SM75
- Tensor parallelism: TP=2
- Model: Qwen2.5-3B sharded_state
- API: OpenAI-compatible vLLM endpoint
- Metrics: vLLM Prometheus /metrics

## Baseline

- vLLM server health: PASS
- /v1/models: PASS
- /v1/chat/completions: PASS
- /metrics: PASS
- TP workers: PASS
- Throttle smoke: PASS

## Throttle smoke

Smoke completed successfully for tested concurrency:
- 1
- 4
- 8

This is smoke evidence only and not a production recommendation.

## validate-sim — unmodified Throttle

Unmodified validate-sim failed during Light load with HTTP 404.

Root cause observed in Throttle 0.3.0:
inconsistent construction of /v1/chat/completions when --endpoint-url
already includes /v1.

## Local validation patch

A local test-only URL fix was applied so validate-sim could proceed.
See:

validate-sim-local-fix.diff

## validate-sim — patched local test

### Light load

- Simulated wall clock: 22.11 s
- Measured wall clock: 24.70 s
- Error: -10.5%

- Simulated input throughput: 204.4 tok/s
- Measured input throughput: 207.2 tok/s
- Error: -1.4%

- Simulated output throughput: 142.2 tok/s
- Measured output throughput: 50.3 tok/s
- Error: +182.8%

- Simulated input cost: $1.36/M input tokens
- Measured normalized input cost: $1.34/M
- Error: +1.4%

Peak measured concurrency: 8

### Medium load

- Simulated wall clock: 18.58 s
- Measured wall clock: 19.96 s
- Error: -6.9%

- Simulated input throughput: 504.7 tok/s
- Measured input throughput: 544.8 tok/s
- Error: -7.4%

- Simulated output throughput: 423.7 tok/s
- Measured output throughput: 154.3 tok/s
- Error: +174.5%

- Simulated input cost: $0.55/M input tokens
- Measured normalized input cost: $0.51/M
- Error: +7.9%

Peak measured concurrency: 50

### Heavy load

Heavy workload failed at request 86 with HTTP 400.

The failure was preserved rather than patched further.

## Additional issue observed

During validate-sim shutdown, concurrent async request tasks emitted:

ValueError:
second argument (exceptions) must be a non-empty sequence

This occurred inside the AnyIO/httpcore connection path during asyncio shutdown.

## Interpretation

On this Turing/SM75 TP=2 environment:

- simulator wall-clock estimates were within roughly 7-11%
- simulator input throughput estimates were within roughly 1-8%
- simulator output throughput was overpredicted by roughly 175-183%

This suggests the current cost/performance simulator does not transfer cleanly
from its existing high-end GPU validation to dual Tesla T4 output-generation
performance.

The $1.00 GPU-hour value used by validate-sim was a synthetic normalization
input required by the CLI and does not represent Kaggle pricing.
EOF

cat "$OUT"

# Throttle validation on Kaggle dual NVIDIA Tesla T4

## Environment

- Throttle: 0.3.0
- kaggle-vllm: 0.1.2
- Backend: upstream-derived vLLM runtime
- GPU: 2 x NVIDIA Tesla T4
- Compute capability: 7.5 / SM75
- Tensor parallelism: TP=2
- Model: Qwen2.5-3B sharded_state
- API: OpenAI-compatible vLLM endpoint
- Metrics: vLLM Prometheus /metrics

## Baseline

- vLLM server health: PASS
- /v1/models: PASS
- /v1/chat/completions: PASS
- /metrics: PASS
- TP workers: PASS
- Throttle smoke: PASS

## Throttle smoke

Smoke completed successfully for tested concurrency:
- 1
- 4
- 8

This is smoke evidence only and not a production recommendation.

## validate-sim — unmodified Throttle

Unmodified validate-sim failed during Light load with HTTP 404.

Root cause observed in Throttle 0.3.0:
inconsistent construction of /v1/chat/completions when --endpoint-url
already includes /v1.

## Local validation patch

A local test-only URL fix was applied so validate-sim could proceed.
See:

validate-sim-loc

### Cell 48 — Copy the local patch to `/kaggle/working`
Places the diff at the output root so it can be downloaded directly from Kaggle's Output panel.

In [68]:
%%bash

cp \
  /kaggle/working/throttle-t4-validation/logs/validate-sim-local-fix.diff \
  /kaggle/working/throttle-validate-sim-url-fix.diff

### Cell 49 — Archive validation evidence
Creates `throttle-kaggle-dual-t4-validation-2026-08-27.tar.gz` plus SHA-256 checksum from the dedicated evidence directory.

In [69]:
%%bash
set -euo pipefail

cd /kaggle/working

ARCHIVE="throttle-kaggle-dual-t4-validation-2026-08-27.tar.gz"

rm -f "$ARCHIVE" "$ARCHIVE.sha256"

tar -czf "$ARCHIVE" \
  throttle-t4-validation

sha256sum "$ARCHIVE" > "$ARCHIVE.sha256"

ls -lh "$ARCHIVE" "$ARCHIVE.sha256"

-rw-r--r-- 1 root root 57K Aug 27 06:24 throttle-kaggle-dual-t4-validation-2026-08-27.tar.gz
-rw-r--r-- 1 root root 119 Aug 27 06:24 throttle-kaggle-dual-t4-validation-2026-08-27.tar.gz.sha256


### Cell 50 — Archive the exact tested source tree
Creates a lightweight source snapshot with the local fix while excluding `.git`, `.venv`, build products, egg-info, and caches, then writes its checksum.

In [70]:
%%bash
set -euo pipefail

cd /kaggle/working

ARCHIVE="throttle-source-tested-with-local-fix.tar.gz"

rm -f "$ARCHIVE" "$ARCHIVE.sha256"

tar \
  --exclude='throttle/.git' \
  --exclude='throttle/.venv' \
  --exclude='throttle/build' \
  --exclude='throttle/src/throttle_pro.egg-info' \
  --exclude='*/__pycache__' \
  -czf "$ARCHIVE" \
  throttle

sha256sum "$ARCHIVE" > "$ARCHIVE.sha256"

ls -lh "$ARCHIVE" "$ARCHIVE.sha256"


-rw-r--r-- 1 root root 584K Aug 27 06:25 throttle-source-tested-with-local-fix.tar.gz
-rw-r--r-- 1 root root  111 Aug 27 06:25 throttle-source-tested-with-local-fix.tar.gz.sha256


### Cell 51 — Write compact environment/provenance evidence
Creates `throttle-kaggle-t4-environment.txt` containing Python, Throttle, kaggle-vllm, PyTorch, NCCL, Triton, `nvidia-smi`, exact upstream commit, and the local diff.

In [71]:
%%bash

OUT=/kaggle/working/throttle-kaggle-t4-environment.txt

{
    date -u

    echo
    python --version

    echo
    throttle --version

    echo
    python -m pip show \
      throttle-pro \
      kaggle-vllm \
      torch \
      nvidia-nccl-cu12 \
      triton

    echo
    nvidia-smi

    echo
    cd /kaggle/working/throttle
    echo "Throttle commit:"
    git rev-parse HEAD

    echo
    echo "Local diff:"
    git diff -- src/throttle/cli.py

} > "$OUT" 2>&1

cat "$OUT"

Thu Aug 27 06:25:43 AM UTC 2026

Python 3.12.13

throttle 0.3.0

Name: throttle-pro
Version: 0.3.0
Summary: Honest load and cost benchmarking for OpenAI-compatible chat endpoints
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: fastapi, httpx, uvicorn
Required-by: 
---
Name: kaggle-vllm
Version: 0.1.2
Summary: A lightweight Kaggle compatibility SDK around upstream vLLM for Tesla T4 GPUs.
Home-page: 
Author: kaggle-vllm contributors
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 
---
Name: torch
Version: 2.10.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-

### Cell 52 — Copy the validation summary to the output root
Copies the summary Markdown to `/kaggle/working/THROTTLE-KAGGLE-T4-VALIDATION-SUMMARY.md` for convenient downloading.

In [73]:
!cp \
  /kaggle/working/throttle-t4-validation/VALIDATION-SUMMARY.md \
  /kaggle/working/THROTTLE-KAGGLE-T4-VALIDATION-SUMMARY.md

### Cell 53 — Empty placeholder cell
This original Kaggle cell contains no code and produced no output. It is retained unchanged.

### Cell 54 — Empty placeholder cell
This original Kaggle cell contains no code and produced no output. It is retained unchanged.

### Cell 55 — Empty placeholder cell
This original Kaggle cell contains no code and produced no output. It is retained unchanged.

### Cell 56 — Empty placeholder cell
This original Kaggle cell contains no code and produced no output. It is retained unchanged.

### Cell 57 — Save the recursive Kaggle working-directory inventory
Runs `ls -laR` into `kaggle-working-directory-files-folders.txt` so all generated runtime/model/source/evidence files are inventoried at session end.

In [66]:
ls -laR > kaggle-working-directory-files-folders.txt

### Cell 58 — Save the final pip package inventory
Writes the complete installed package/version list to `kaggle-pip-list.txt`.

In [64]:
!pip list > kaggle-pip-list.txt

### Cell 59 — Empty final placeholder cell
This original Kaggle cell contains no code and produced no output. It is retained unchanged.